In [3]:
%matplotlib widget

import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import random
import math
import operator

In [4]:
df = pd.read_csv("iris.csv")
display(df.head())
display(df.shape)

FEATURES = df.columns.values.tolist()
PREDICT = FEATURES.pop()
# display(FEATURES)

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


(150, 5)

In [5]:
# get 70% train and 30% test
msk = np.random.rand(150) < 0.7     # display(msk)
train = df[msk]
test = df[~msk]
display(test.shape)

(52, 5)

## **Bagging and Random Forest**

In [6]:
class Node():
    def __init__(self, l=None, r=None, label=None, m=None):
        self.median = m
        self.left = l
        self.right = r
        self.label = label

    def leaf(self):
        return self.left==None and self.right==None

In [7]:
def get_split(data, feat):
    md = data[feat].median()
    left = data[data[feat] <= md]
    right = data[data[feat] > md]
    # print(left, right)
    return left, right

def get_frequency(data, feature):
    return len(data[feature]), dict(data[feature].value_counts())


### **Information**
$$I(X) = log(1/p(x)) = -log(p(x))$$

### **Entropy**
$$H(X) = E[I(X)] = \sum_{f}^{features}p(f)I(f) = -\sum_{f}^{features}p(f)log(f)$$

#### **Disorder in a set**

$$D_s = - \sum_{c \in C} P_c log_2(P_c)$$

### **Information gain**

$$Gain(S) = D_s(S) - \sum_{f \in features}  \frac{|S_f|}{|S|} * D_s(S_f)$$
$$\text{Entropía global} - \text{entropía ponderada de las clases}$$

In [8]:
def get_entropy(data):
    ln, classes = get_frequency(data, PREDICT)
    return -np.sum([v/ln * np.log2(v/ln) for k, v in classes.items()])

def get_info_gain(data):    # en Info-Gain se devuelve el del de mayor coeficiente
    ln = len(data[PREDICT])
    global_entropy = get_entropy(data)
    weighted_entropy = 0
    info = {}

    for f in FEATURES:
        l_group, r_group = get_split(data, f)
        weighted_entropy = len(l_group)/ln * get_entropy(l_group) + len(r_group)/ln * get_entropy(r_group)
        info[f] =  global_entropy - weighted_entropy
        # print(f'{f} : {global_entropy} - {weighted_entropy} = {global_entropy - weighted_entropy}')
    # print(info)
        
    # getting key(feature) with maximum value(info_gain) in dictionary
    return max(info.items(), key=operator.itemgetter(1))[0]     


### **Gini index**

$$Gini(x) = 1 - \sum_{h}^{hijos} [p(h|padre)]^2$$
$$p(j|t) \text{ es la frecuencia relativa de la clase j en el nodo t}$$

### **Gini split**

$$Gini_{split} = \sum_{i=1}^{k} \frac{n_i}{n} Gini(i)$$

$$n_i \text{ es el número de registros en el hijo i, n es el 
número de registros en el nodo p}$$

In [9]:
def get_gini(data):
    ln, classes = get_frequency(data, PREDICT)
    return 1 - np.sum([(v/ln)**2 for k, v in classes.items()])

def gini_split(data):       # en Gini se devuelve el del minimo valor
    ln = len(data[PREDICT])
    gini = {}

    for f in FEATURES:
        l_group, r_group = get_split(data, f)
        gini[f] = len(l_group)/ln * get_gini(l_group) + len(r_group)/ln * get_gini(r_group)
    # print(gini)
    return min(gini.items(), key=operator.itemgetter(1))[0]     

In [10]:
class Tree():
    def __init__(self, h):
        self.root = None
        self.heuristic = h
    
    def build(self, data):
        # get classes of current node
        classes = np.unique(data[PREDICT])     
        
        # stopping criteria
        if len(classes) == 1:
            # return leaf node because data is pure
                # print(f'\tc: {classes}')
            return Node(label=classes[0])       
        else:
            # select attribute B according heuristic_function
            best_feature = self.heuristic(data)     
                # print(f'\t{classes} \t{best_feature}')

            # generate left and right nodes until they become pure
            l_group, r_group = get_split(data, best_feature)
                # print("L")
            left_tree = self.build(l_group)
                # print("R")
            right_tree = self.build(r_group)
            return Node(l=left_tree, r=right_tree, label=best_feature, m=data[best_feature].median())
    
    def train(self, data):
        self.root = self.build(data)
    
    def test(self, data):
        bool_list = [self.traversal(self.root, d)==d[PREDICT] for d in data.to_dict(orient='records')]
        return 100*sum(bool_list)/len(bool_list)
    
    def predict(self, row):
        return self.traversal(self.root, row) == row[PREDICT]

    def traversal(self, node, row):
        if node.leaf():
            return node.label
        elif row[node.label] <= node.median:
            return self.traversal(node.left, row)
        else:
            return self.traversal(node.right, row)

In [11]:
def printTree(node, level=0):
    if node != None:
        printTree(node.left, level + 1)
        print(' ' * 4 * level + '-> ' + node.label)
        printTree(node.right, level + 1)

In [12]:
t_info = Tree(get_info_gain)
t_info.train(train)
print(f'{round(t_info.test(test), 5)}% accuracy\n')
# printTree(t_info.root)

96.15385% accuracy



In [13]:
t_gini = Tree(gini_split)
t_gini.train(train)
print(f'{round(t_gini.test(test), 5)}% accuracy\n')
# printTree(t_gini.root)

96.15385% accuracy



## **Bootstrap aggregating**

In [50]:
def random_forest(data, n):
    forest = []
    for _ in range(n):
        msk = np.random.rand(150) < 0.7     # display(msk)
        tr = Tree(get_info_gain)
        tr.train(df[msk])
        forest.append(tr)
    return forest

In [51]:
def predict(trees, row):
    # print(row)
    pred = [t.traversal(t.root, row) for t in trees]
    # print(pred)
    return max(set(pred), key=pred.count)

In [52]:
def bootstrap(trees, data):
    bool_list = [predict(trees, d) == d[PREDICT] for d in data.to_dict(orient='records')]
    # print(bool_list)
    return 100*sum(bool_list)/len(bool_list)

In [53]:
bootstrap(random_forest(train, 5), test)
# https://utec.zoom.us/rec/play/Ii1joIGWEameQPPQJwkGCdR9DurbWuy3nJ1o_uQ0mlkfAvacLjrZ4wSrD4ErXawURCy7SpuTXEaeuQQM.0X4XllkPLk2XeXOv

100.0

### **References**
- [probabilidad condicional](https://es.wikipedia.org/wiki/Probabilidad_condicionada)
- [media mediana y varianza]()
- [bagging bootstrap aggregating]()
- [bias in ML](https://www.bmc.com/blogs/bias-variance-machine-learning/)
- [decision tree sklearn](https://scikit-learn.org/stable/auto_examples/tree/plot_iris_dtc.html)
- [get list from pd.Df column or row](https://stackoverflow.com/questions/22341271/get-list-from-pandas-dataframe-column-or-row)
- [iterate over all or certain](https://thispointer.com/pandas-loop-or-iterate-over-all-or-certain-columns-of-a-dataframe/)